# Assignment 9: Pretrained Vision Transformer vs CNN
Fine-tune a pretrained **ViT-B/16** and a pretrained **ResNet50** (CNN) on the same CIFAR-10 subset and compare accuracy, speed and size.

In [ ]:
!pip install -q transformers
import time, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from transformers import ViTForImageClassification
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

torch.manual_seed(42); np.random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"; print(device)

tf_ = transforms.Compose([transforms.Resize(224), transforms.ToTensor(), transforms.Normalize([0.5]*3, [0.5]*3)])  # ViT-21k norm
tf_cnn = transforms.Compose([transforms.Resize(224), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

In [ ]:
N_TRAIN, N_TEST = 5000, 1000
tr_full_v = datasets.CIFAR10("./data", True, transform=tf_, download=True); te_full_v = datasets.CIFAR10("./data", False, transform=tf_, download=True)
tr_full_c = datasets.CIFAR10("./data", True, transform=tf_cnn); te_full_c = datasets.CIFAR10("./data", False, transform=tf_cnn)
tr_idx = np.random.permutation(len(tr_full_v))[:N_TRAIN]; te_idx = np.random.permutation(len(te_full_v))[:N_TEST]
mk = lambda ds, idx, sh: DataLoader(Subset(ds, idx), batch_size=32 if sh else 64, shuffle=sh, num_workers=2)
dl_vit = (mk(tr_full_v, tr_idx, True), mk(te_full_v, te_idx, False))
dl_cnn = (mk(tr_full_c, tr_idx, True), mk(te_full_c, te_idx, False))
classes = tr_full_v.classes

In [ ]:
class ViTWrap(nn.Module):
    def __init__(self):
        super().__init__()
        self.m = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224-in21k", num_labels=10)
    def forward(self, x): return self.m(pixel_values=x).logits

def vit_model():
    m = ViTWrap()
    for n, p in m.named_parameters():                 # freeze all but last encoder block + classifier
        p.requires_grad = ("encoder.layer.11" in n) or ("classifier" in n) or ("layernorm" in n)
    return m.to(device)

def cnn_model():
    m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    for n, p in m.named_parameters():                 # freeze all but last residual stage + fc
        p.requires_grad = n.startswith("layer4")
    m.fc = nn.Linear(m.fc.in_features, 10)
    return m.to(device)

def run(name, model, dls, epochs=3, lr=3e-5):
    tr, te = dls
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr); crit = nn.CrossEntropyLoss(); hist = []
    t0 = time.time()
    for ep in range(epochs):
        model.train(); tot = 0
        for x, y in tr:
            x, y = x.to(device), y.to(device); opt.zero_grad(); loss = crit(model(x), y); loss.backward(); opt.step(); tot += loss.item() * len(x)
        hist.append(tot / len(tr.dataset)); print(f"{name} epoch {ep+1} loss {hist[-1]:.4f}")
    train_t = time.time() - t0
    model.eval(); yt, yp = [], []; t1 = time.time()
    with torch.no_grad():
        for x, y in te: yp += model(x.to(device)).argmax(1).cpu().tolist(); yt += y.tolist()
    infer_t = (time.time() - t1) / len(yt) * 1000
    return dict(Model=name, Accuracy=accuracy_score(yt, yp), MacroF1=f1_score(yt, yp, average="macro"),
                TotalParams_M=sum(p.numel() for p in model.parameters())/1e6, TrainableParams_M=sum(p.numel() for p in params)/1e6,
                TrainTime_s=train_t, InferMs_per_img=infer_t), hist, yt, yp

r_vit, h_vit, yt_v, yp_v = run("ViT-B/16", vit_model(), dl_vit, lr=3e-5)
r_cnn, h_cnn, yt_c, yp_c = run("ResNet50 (CNN)", cnn_model(), dl_cnn, lr=1e-4)
res = pd.DataFrame([r_vit, r_cnn]).set_index("Model"); res.round(4)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(h_vit, marker="o", label="ViT"); ax[0].plot(h_cnn, marker="o", label="ResNet50"); ax[0].set_title("Training loss"); ax[0].legend()
for a, (t, p, n) in zip(ax[1:], [(yt_v, yp_v, "ViT"), (yt_c, yp_c, "ResNet50")]):
    sns.heatmap(confusion_matrix(t, p), annot=True, fmt="d", cmap="Blues", ax=a, cbar=False, xticklabels=classes, yticklabels=classes)
    a.set_title(f"{n} confusion matrix"); a.tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.show()

## Comparison
- **CNN (ResNet50):** convolutions have a built-in locality bias, so it learns well from small data and is cheaper to run.
- **ViT:** splits the image into 16x16 patches and uses self-attention to model global relationships; it needs large-scale pre-training (here ImageNet-21k) but transfers very well and often wins when fine-tuned, at higher compute cost.
Check the table above for this run's accuracy, parameters and speed.